In [3]:
import pandas as pd
import re

# 读取你刚才保存的垃圾桶数据
df_rej = pd.read_csv('../data/processed/stage1_rejected_data.csv')

# 提取核心文本并计算长度
df_rej['core_text'] = df_rej['Reply'].astype(str).apply(lambda x: re.sub(r'^(?:尊敬的.*?您好[,，。！\s]*|感谢.*?关注[,，。！\s]*)+|(?:感谢.*?关注|请以.*?为准|谢谢)[,，。！\s]*$', '', x).strip())
df_rej['core_len'] = df_rej['core_text'].apply(len)

# 抽样查看：长度大于 20，却被我们误杀的到底是什么内容？
samples = df_rej[df_rej['core_len'] > 20].sample(500, random_state=42)
for idx, row in samples.iterrows():
    print(f"[{row['core_len']}字] {row['core_text']}")
    print("-" * 50)

[47字] 您好！相关内容敬请查阅公司年度报告“第十节财务报告”之五“重要会计政策及会计估计”的相关内容。
--------------------------------------------------
[44字] 您好，我行2021年半年度报告中计入交易性金融资产中的基金投资主要为货币基金和债券基金。
--------------------------------------------------
[37字] 您好，截至3月31日，公司股东人数为9,370户。谢谢您对公司的关注支持！
--------------------------------------------------
[24字] 感谢您的提问！公司目前没有涉足新型冠状病毒检测。
--------------------------------------------------
[76字] 您好，股价运行有其内在规律和价值发现特点，并受大盘涨跌及资金面等多重因素影响。公司控股股东安钢集团混合所有制改革相关信息，请查阅公司相关公告，谢谢关注！
--------------------------------------------------
[79字] 您好，公司已经建立规范的财务管理架构和制度，目前使用ERP系统和OA协同集成办公软件，满足公司对财务管理的要求。未来，公司将结合业务发展进一步提升信息化水平。
--------------------------------------------------
[52字] 您好，万东积极履行社会责任，第一时间将公司产品运抵一线医院，加班加点保障设备的供应等等，为生命保驾护航。
--------------------------------------------------
[108字] 公司将在定期报告中按要求对对应时点的股东信息进行披露；作为公司股东，如需查询特定时点的股东人数，需将查询日个人身份证明文件、持股证明以及联系方式发送至公司邮箱Info@demxs.com，公司核实股东身份后予以回复。
--------------------------------------------------
[27字] 您好，zaful所售产品不在近期美国豁免关税的清单里。
----------------------------------

分析语义过滤率偏低的原因：
我们可以用极端的数学逻辑来拆解一下你的得分机制，问题出在阈值设定过于宽松（MIN_FINAL_SCORE = 0.18）以及TF-IDF 文本向量的稀疏性上：

1、绝对保底分数过高：
哪怕董秘给了一个极其冗长且毫无营养的回答，你的 length_score 最低也会给 0.4。
BETA_LENGTH_NORM 权重是 0.35，这意味着它保底能拿到 0.4 * 0.35 = 0.14 分。

2、“伪新词”拉高了信息增益：
中文分词后，即便董秘只是在打太极，也会引入大量提问中没有的副词、连词或企业标准 PR 词汇（如“始终坚持”、“积极推进”）。只要这些词占了 20%，new_word_ratio 就能拿到 0.2。
此时的信息增益得分为：0.14 + (0.65 * 0.2) = 0.27。

3、TF-IDF 相似度的天然特性：
短文本的 TF-IDF 矩阵是非常稀疏的。除非董秘做到 100% 复制粘贴散户的提问，否则余弦相似度很难超过 0.5。假设相似度为 0.3，那么惩罚系数就是 (1 - 0.3) = 0.7。
最终得分：0.27 * 0.7 = 0.189。

In [5]:
import pandas as pd

# 读取你刚刚跑出来的 71 万条有效数据
df = pd.read_csv("../data/processed/stage2_entropy_filtered_data.csv")

# 1. 查看分数的统计学分布
print("===== 最终得分统计 =====")
print(df['final_score'].describe())

# 2. 查看不同分位数的切断点
print("\n===== 分位数切割点 =====")
print(df['final_score'].quantile([0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.50]))

===== 最终得分统计 =====
count    713349.000000
mean          0.778695
std           0.145475
min           0.180007
25%           0.700915
50%           0.805958
75%           0.888466
max           1.000000
Name: final_score, dtype: float64

===== 分位数切割点 =====
0.01    0.327023
0.05    0.494127
0.10    0.578528
0.15    0.630640
0.20    0.670110
0.25    0.700915
0.30    0.726234
0.50    0.805958
Name: final_score, dtype: float64


In [10]:
import pandas as pd

# 读取数据
df = pd.read_csv("../data/processed/stage2_entropy_filtered_data.csv")

def spot_check(df, min_score, max_score, sample_num=5):
    print(f"\n==== 抽查分数区间: {min_score} - {max_score} ====")
    samples = df[(df['final_score'] >= min_score) & (df['final_score'] <= max_score)]
    if len(samples) == 0:
        return
    for idx, row in samples.sample(min(sample_num, len(samples)), random_state=42).iterrows():
        print(f"得分: {row['final_score']:.3f}")
        print(f"Q: {row['Qsubj']}")
        print(f"A: {row['Reply']}\n{'-'*40}")

# 1. 抽查极度疑似废话的区间 (约 5% 分位数)
spot_check(df, 0.45, 0.50)

# 2. 抽查疑似边界的区间 (约 10% - 15% 分位数)
spot_check(df, 0.58, 0.63)

# 3. 抽查可能被误杀的区间 (约 20% 分位数)
spot_check(df, 0.65, 0.68)


==== 抽查分数区间: 0.45 - 0.5 ====
得分: 0.495
Q: 贵公司的应收账款收现天数比较长，有收不回的风险，贵公司有哪些改善措施？
A: 尊敬的投资者，您好！公司应收账款有一定增幅，主要有两方面原因：一是公司新能源板块因可再生能源电价补贴款回款周期较长，应收补贴款持续增加，造成应收账款收现天数较长；二是公司民爆板块随着爆破一体化深入推进，年度工程爆破收入持续增加，因业主方货款结算期较长造成收现天数有所延长。公司结合历史回款情况，经综合研判，公司应收账款均属良性资产，不存在无法收回的风险。公司将采取以下改善措施：一是及时跟踪国家可再生基金补助资金的最新动态，积极申报相关材料，争取新能源产业政府补贴回款，加快清欠款项的收回；二是进一步加强对工程爆破业务的款项回收力度，按照合同约定期限，积极与业主沟通，定期回收、催收爆破服务款项。
----------------------------------------
得分: 0.458
Q: 请问公司间接持有的侨银环保和小鸟看看的股份数量分别是多少，谢谢
A: 您好，截至目前，公司通过珠海广发信德环保产业投资基金合伙企业（有限合伙）间接持有侨银城市管理股份有限公司股份计14,762,841股，公司间接持有的小鸟看看股份已全部减持完毕。感谢您的关注。
----------------------------------------
得分: 0.486
Q: 您好，我想问下贵公司的融资渠道有哪些？
A: 您好，公司融资渠道主要有银行贷款、再融资等。感谢您的关注。
----------------------------------------
得分: 0.462
								贵司有没计划参与广东省集成电路产业基金？
A: 您好，公司目前暂无参与广东省集成电路产业基金计划。如有相关事项，公司将严格按照有关法律法规的规定和要求，及时履行信息披露义务。感谢您对公司的关注，谢谢！
----------------------------------------
得分: 0.480
Q: 公司高溢价收购博林达是否存在利益输送？
A: 公司本次收购博林达价格合理，不存在利益输送。公司对博林达相关经济效益指标合理预测，聘请专业的第三方审计和评估机构提供定价依据。谢谢！
-----------------------